# mae — video ingestion via Qwen2.5-VL (Deepnote-triggered)

Per Task #174 (2026-05-03 spec, session `24cbed9c`). Replaces the RunPod L40S video-extraction path with a Deepnote-Jobs-API trigger: POST → notebookRunId → poll → JSON output to `/work/output.json` → `<downstream-consumer>` ingests.

**Pipeline**: `yt-dlp` → frame extraction (every `FRAME_INTERVAL_SEC` seconds, max `MAX_FRAMES`) → `Qwen2.5-VL` inference per frame with a trading-intelligence prompt → aggregated signal record → `/work/output.json`.

**Parameters** (override via Deepnote Job `parameters` payload):
- `VIDEO_URL` — full YouTube URL or video ID
- `EXTRACT_MODE` — `frames` (visual analysis), `transcript` (audio/captions only), or `both`
- `FRAME_INTERVAL_SEC` — seconds between sampled frames (default 12)
- `MAX_FRAMES` — cap on frames to limit GPU cost (default 30)
- `MODEL` — Qwen2.5-VL model handle (default `Qwen/Qwen2.5-VL-7B-Instruct`; use `-3B-Instruct` on small GPUs)
- `PROMPT_PROFILE` — `trading-intelligence` (default) | `general-summary` | `custom`
- `OUTPUT_PATH` — where to write JSON (default `/work/output.json`)

**GPU**: Deepnote's GPU tier (T4 / A10 / A100 depending on availability). 7B-Instruct fits on T4 (16GB) in 8-bit; 3B-Instruct comfortably in FP16. Falls back to CPU if no GPU — slow but functional for the 3B model.

**Trigger via Deepnote API** — see `README.md` in this directory for the curl pattern + Python wrapper.

**Out of scope here**: signal scoring + position-sizing — `<downstream-consumer>` consumes this JSON and routes to the orchestrator gate stack.

In [ ]:
# §1 Setup — install + import
import os, sys, json, subprocess, time, base64, hashlib, re
from pathlib import Path
from datetime import datetime, timezone

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *pkgs], check=True)

# yt-dlp + ffmpeg (frame extraction) + transformers/qwen-vl deps
try:
    import yt_dlp
except ImportError:
    _pip('yt-dlp')
    import yt_dlp

# ffmpeg binary check (frame extraction). Deepnote has it; bare-CPU containers may not.
ffmpeg_ok = subprocess.run(['ffmpeg', '-version'], capture_output=True).returncode == 0
if not ffmpeg_ok:
    print('Installing ffmpeg (apt)...')
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)

# Vision-language deps — defer to §4 (only load if EXTRACT_MODE != transcript-only)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'torch device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory // 1024**3} GB)')

WORK = Path('/work') if Path('/work').exists() else Path('/tmp/mae-video-work')
WORK.mkdir(parents=True, exist_ok=True)
FRAMES_DIR = WORK / 'frames'
FRAMES_DIR.mkdir(exist_ok=True)
print(f'WORK: {WORK}')

In [ ]:
# §2 Parameters — Deepnote Jobs API injects these at trigger time.
# Defaults here let the notebook run standalone for manual testing.

VIDEO_URL          = os.environ.get('VIDEO_URL',          '')  # set by Deepnote Job parameters
EXTRACT_MODE       = os.environ.get('EXTRACT_MODE',       'both')           # frames | transcript | both
FRAME_INTERVAL_SEC = int(os.environ.get('FRAME_INTERVAL_SEC', '12'))
MAX_FRAMES         = int(os.environ.get('MAX_FRAMES',         '30'))
MODEL              = os.environ.get('MODEL',              'Qwen/Qwen2.5-VL-7B-Instruct')
PROMPT_PROFILE     = os.environ.get('PROMPT_PROFILE',     'trading-intelligence')
OUTPUT_PATH        = Path(os.environ.get('OUTPUT_PATH',  str(WORK / 'output.json')))

if not VIDEO_URL:
    # Standalone test fallback — operator should override via Deepnote Job params.
    # When operator provides a YT URL the live trigger sets VIDEO_URL.
    raise ValueError(
        'VIDEO_URL is required. Set it via Deepnote Job parameters: '
        '{"VIDEO_URL": "https://youtu.be/<id>"}'
    )

video_id_match = re.search(r'(?:v=|youtu\.be/|/watch\?v=)([\w-]{11})', VIDEO_URL) or re.match(r'^([\w-]{11})$', VIDEO_URL)
VIDEO_ID = video_id_match.group(1) if video_id_match else hashlib.md5(VIDEO_URL.encode()).hexdigest()[:11]

print(f'VIDEO_URL:           {VIDEO_URL}')
print(f'VIDEO_ID:            {VIDEO_ID}')
print(f'EXTRACT_MODE:        {EXTRACT_MODE}')
print(f'FRAME_INTERVAL_SEC:  {FRAME_INTERVAL_SEC}')
print(f'MAX_FRAMES:          {MAX_FRAMES}')
print(f'MODEL:               {MODEL}')
print(f'PROMPT_PROFILE:      {PROMPT_PROFILE}')
print(f'OUTPUT_PATH:         {OUTPUT_PATH}')

## §3 yt-dlp — download video + extract transcript

Downloads at 360p (small footprint, sufficient for VL frame analysis). Captures captions when available (faster than whisper); falls back to no-transcript if EXTRACT_MODE excludes transcript.

In [ ]:
VIDEO_PATH = WORK / f'{VIDEO_ID}.mp4'
SUBS_PATH = WORK / f'{VIDEO_ID}.en.vtt'

ydl_opts = {
    'format': 'best[height<=360][ext=mp4]/best[height<=360]/best',
    'outtmpl': str(VIDEO_PATH.with_suffix('')) + '.%(ext)s',
    'writesubtitles': EXTRACT_MODE in ('transcript', 'both'),
    'writeautomaticsub': EXTRACT_MODE in ('transcript', 'both'),
    'subtitleslangs': ['en'],
    'subtitlesformat': 'vtt',
    'quiet': True,
    'no_warnings': True,
}

print(f'Downloading {VIDEO_URL}...')
t0 = time.time()
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(VIDEO_URL, download=True)
duration_sec = info.get('duration', 0)
title = info.get('title', '')
uploader = info.get('uploader', '')
print(f'  title:     {title}')
print(f'  uploader:  {uploader}')
print(f'  duration:  {duration_sec}s')
print(f'  download:  {time.time()-t0:.1f}s')

# Parse transcript if available
transcript_text = ''
transcript_segments = []
if EXTRACT_MODE in ('transcript', 'both') and SUBS_PATH.exists():
    raw = SUBS_PATH.read_text()
    # Minimal WebVTT parser — extract timestamps + text
    block_re = re.compile(r'(\d{2}:\d{2}:\d{2}\.\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}\.\d{3})[^\n]*\n(.*?)(?=\n\n|\Z)', re.DOTALL)
    def _ts_to_sec(s):
        h, m, rest = s.split(':')
        return int(h)*3600 + int(m)*60 + float(rest)
    for m in block_re.finditer(raw):
        seg_text = re.sub(r'<[^>]+>', '', m.group(3)).strip().replace('\n', ' ')
        if seg_text:
            transcript_segments.append({
                'start_sec': _ts_to_sec(m.group(1)),
                'end_sec':   _ts_to_sec(m.group(2)),
                'text':      seg_text,
            })
            transcript_text += seg_text + ' '
    transcript_text = transcript_text.strip()
    print(f'  transcript: {len(transcript_segments)} segments, {len(transcript_text)} chars')
else:
    print('  transcript: (skipped or unavailable)')

## §3b Frame extraction via ffmpeg

Sample one frame every `FRAME_INTERVAL_SEC` seconds, capped at `MAX_FRAMES`. Saves to `WORK/frames/` as JPEG.

In [ ]:
frame_paths = []
if EXTRACT_MODE in ('frames', 'both'):
    # Compute frame timestamps (capped at MAX_FRAMES)
    n_possible = max(1, duration_sec // FRAME_INTERVAL_SEC)
    n_frames = min(MAX_FRAMES, n_possible)
    if n_possible > MAX_FRAMES:
        # Distribute MAX_FRAMES evenly across duration (don't just take the first MAX_FRAMES)
        step = duration_sec / MAX_FRAMES
        timestamps = [step * i for i in range(MAX_FRAMES)]
    else:
        timestamps = [FRAME_INTERVAL_SEC * i for i in range(n_frames)]

    print(f'Extracting {len(timestamps)} frames at intervals of ~{duration_sec/max(len(timestamps),1):.1f}s...')
    for i, ts in enumerate(timestamps):
        out = FRAMES_DIR / f'frame_{i:03d}_{int(ts):05d}s.jpg'
        subprocess.run(['ffmpeg', '-y', '-ss', f'{ts:.2f}', '-i', str(VIDEO_PATH),
                        '-frames:v', '1', '-q:v', '3', str(out)],
                       check=True, capture_output=True)
        frame_paths.append({'idx': i, 'timestamp_sec': ts, 'path': str(out)})
    print(f'  extracted: {len(frame_paths)} frames')
else:
    print('Frame extraction skipped (EXTRACT_MODE=transcript)')

## §4 Qwen2.5-VL model load

Loads the model with whatever precision fits the available GPU memory:
- **A100 40GB / A10 24GB**: 7B-Instruct in FP16 (~16 GB)
- **T4 16GB**: 7B-Instruct in 8-bit (~10 GB) via `bitsandbytes`
- **<16GB or CPU**: caller should override `MODEL=Qwen/Qwen2.5-VL-3B-Instruct`

Skip this cell entirely if `EXTRACT_MODE=transcript`.

In [ ]:
vl_model = None
vl_processor = None

if EXTRACT_MODE in ('frames', 'both'):
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    except ImportError:
        _pip('transformers>=4.49', 'accelerate', 'qwen-vl-utils', 'bitsandbytes')
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

    load_kwargs = {'torch_dtype': torch.float16, 'device_map': 'auto'}
    if DEVICE == 'cuda':
        gpu_gb = torch.cuda.get_device_properties(0).total_memory // 1024**3
        if gpu_gb < 20 and '7B' in MODEL:
            print(f'GPU has {gpu_gb} GB — loading in 8-bit')
            load_kwargs['load_in_8bit'] = True
            load_kwargs.pop('torch_dtype', None)
    print(f'Loading {MODEL} on {DEVICE}...')
    t0 = time.time()
    vl_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL, **load_kwargs)
    vl_processor = AutoProcessor.from_pretrained(MODEL)
    print(f'  loaded in {time.time()-t0:.1f}s')

## §5 Inference — per-frame structured extraction

Per the `trading-intelligence` prompt profile, each frame returns:
- `tickers` — symbols mentioned or visible (crypto/equity/forex)
- `price_callouts` — specific price levels referenced
- `sentiment` — bullish/bearish/neutral per mentioned asset
- `macro_events` — FOMC, CPI prints, earnings, geopolitical
- `chart_patterns` — visible chart formations
- `speaker_confidence` — high/medium/low
- `time_sensitivity` — now/this-week/longer-term

In [ ]:
PROMPT_PROFILES = {
    'trading-intelligence': '''You are a market-intelligence analyst watching a financial-content video for an algorithmic trading system.

For THIS FRAME, return STRICT JSON with these exact fields. Use empty arrays / null for fields with no signal. Do not invent.

{
  "tickers": ["BTC", "ETH", ...],
  "price_callouts": [{"ticker": "BTC", "price": 67500, "context": "..."}],
  "sentiment": {"BTC": "bullish|bearish|neutral", ...},
  "macro_events": ["FOMC Wed", "CPI Thu", ...],
  "chart_patterns": ["BTC head-and-shoulders", ...],
  "speaker_confidence": "high|medium|low|null",
  "time_sensitivity": "now|this-week|longer-term|null",
  "frame_notes": "one-sentence summary of what is visible"
}

Return ONLY the JSON object, no preamble.''',
    'general-summary': '''Describe what is visible in this frame in 2 sentences. Return JSON {"summary": "..."}.'''
}
PROMPT = PROMPT_PROFILES.get(PROMPT_PROFILE, PROMPT_PROFILES['trading-intelligence'])

def _infer_one(image_path):
    """Run Qwen2.5-VL on a single frame; return parsed dict or {raw: ...} on parse failure."""
    from PIL import Image
    image = Image.open(image_path).convert('RGB')
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT},
    ]}]
    text = vl_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = vl_processor(text=[text], images=[image], padding=True, return_tensors='pt').to(vl_model.device)
    with torch.inference_mode():
        out_ids = vl_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    out_text = vl_processor.batch_decode(
        out_ids[:, inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )[0].strip()
    try:
        # Extract first JSON object from response (model may include trailing text)
        match = re.search(r'\{.*\}', out_text, re.DOTALL)
        return json.loads(match.group(0)) if match else {'raw': out_text}
    except (json.JSONDecodeError, AttributeError):
        return {'raw': out_text, 'parse_error': True}

frame_signals = []
if vl_model is not None and frame_paths:
    print(f'Inferring {len(frame_paths)} frames...')
    t0 = time.time()
    for fp in frame_paths:
        try:
            sig = _infer_one(fp['path'])
        except Exception as e:
            sig = {'error': str(e)}
        frame_signals.append({
            'frame_idx': fp['idx'],
            'timestamp_sec': fp['timestamp_sec'],
            'signal': sig,
        })
        if (len(frame_signals) % 5) == 0:
            print(f'  {len(frame_signals)}/{len(frame_paths)}  ({time.time()-t0:.1f}s)')
    print(f'Inference complete: {len(frame_signals)} frames in {time.time()-t0:.1f}s')

## §6 Aggregate — signal record for <downstream-consumer>

Per-frame signals are aggregated into a single record that captures: dominant tickers, net sentiment, macro events, time-sensitivity histogram, and the raw per-frame array (for the coach to re-inspect any frame's context).

In [ ]:
from collections import Counter, defaultdict

def _aggregate(frames):
    ticker_counter = Counter()
    sentiment_by_ticker = defaultdict(Counter)
    macro = Counter()
    time_sens = Counter()
    speaker_conf = Counter()
    price_callouts = []
    for f in frames:
        s = f.get('signal', {})
        if not isinstance(s, dict): continue
        for t in s.get('tickers', []) or []:
            ticker_counter[str(t).upper()] += 1
        for t, sent in (s.get('sentiment', {}) or {}).items():
            sentiment_by_ticker[str(t).upper()][str(sent).lower()] += 1
        for ev in s.get('macro_events', []) or []:
            macro[str(ev)] += 1
        ts = s.get('time_sensitivity')
        if ts: time_sens[str(ts).lower()] += 1
        sc = s.get('speaker_confidence')
        if sc: speaker_conf[str(sc).lower()] += 1
        for pc in s.get('price_callouts', []) or []:
            if isinstance(pc, dict):
                pc['frame_idx'] = f.get('frame_idx')
                price_callouts.append(pc)
    # Net sentiment per ticker as bullish_frac - bearish_frac
    net_sent = {}
    for t, c in sentiment_by_ticker.items():
        total = sum(c.values())
        if total:
            net_sent[t] = round((c.get('bullish', 0) - c.get('bearish', 0)) / total, 3)
    return {
        'dominant_tickers': [t for t, _ in ticker_counter.most_common(10)],
        'ticker_frame_counts': dict(ticker_counter),
        'net_sentiment': net_sent,
        'macro_events_mentioned': dict(macro),
        'time_sensitivity_dist': dict(time_sens),
        'speaker_confidence_dist': dict(speaker_conf),
        'price_callouts': price_callouts,
    }

agg = _aggregate(frame_signals) if frame_signals else {}

record = {
    'schema_version': 1,
    'source': f'youtube/{VIDEO_ID}',
    'source_url': VIDEO_URL,
    'title': title,
    'uploader': uploader,
    'duration_sec': duration_sec,
    'extracted_at': datetime.now(timezone.utc).isoformat(),
    'extract_mode': EXTRACT_MODE,
    'model': MODEL,
    'prompt_profile': PROMPT_PROFILE,
    'frames_analyzed': len(frame_signals),
    'transcript': {
        'text': transcript_text,
        'segments': transcript_segments,
        'segment_count': len(transcript_segments),
        'char_count': len(transcript_text),
    },
    'aggregate': agg,
    'frames': frame_signals,  # raw per-frame for coach re-inspection
}

print(f'\n=== Aggregate ===')
print(f'Dominant tickers:   {agg.get("dominant_tickers", [])}')
print(f'Net sentiment:      {agg.get("net_sentiment", {})}')
print(f'Macro events:       {list((agg.get("macro_events_mentioned", {}) or {}).keys())[:5]}')
print(f'Time-sensitivity:   {agg.get("time_sensitivity_dist", {})}')
print(f'Speaker confidence: {agg.get("speaker_confidence_dist", {})}')
print(f'Price callouts:     {len(agg.get("price_callouts", []))}')

## §7 Write output JSON

Lands at `OUTPUT_PATH` (default `/work/output.json`). Deepnote keeps `/work/*` accessible after the job; the trigger script (`README.md`) fetches this file via Deepnote's File API or via committed Git artifact.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(record, indent=2, default=str))
size_kb = OUTPUT_PATH.stat().st_size // 1024
print(f'Wrote {size_kb} KB → {OUTPUT_PATH}')
print()
print(f'JSON preview (first 1.5 KB):')
print(OUTPUT_PATH.read_text()[:1500])